<div style="background: linear-gradient(135deg, #022c22 0%, #064e3b 45%, #0e7490 100%); padding: 40px 30px; border-radius: 16px; margin-bottom: 20px; border: 1px solid #14b8a6; box-shadow: 0 20px 40px rgba(0,0,0,0.3);">
    <div style="text-align: center;">
        <h1 style="color: #f0fdfa;text-align: center; font-size: 36px; margin: 0; letter-spacing: 1px;">EV Adoption Prediction</h1>
        <p style="color: #99f6e4;text-align: center; font-size: 18px; margin-top: 8px; font-weight: 300;">Kaggle Playground Series - Season 6 Episode 9</p>
        <hr style="border: none;text-align: center; height: 3px; background: linear-gradient(90deg, transparent, #10b981, #22d3ee, #a78bfa, transparent); margin: 20px auto; width: 80%;">
        <p style="color: #e2e8f0;text-align: center; font-size: 15px; margin: 0;"><strong style="color: #34d399;">GPU XGBoost</strong> with <strong style="color: #22d3ee;">Neighbourhood Window Encodings</strong> + <strong style="color: #a78bfa;">Triple Target Encoding</strong></p>
    </div>
</div>

### The Idea in One Paragraph

Playground data is generated by a CTGAN, so the numeric axes (income, commute distance)
carry *generator fingerprints* — value spikes, dead zones, cliffs — and the relationship
between a continuous value and the target is locally smooth, not piecewise-constant.
Instead of only feeding the raw numbers to the trees, we encode the two continuous axes
at multiple neighbourhood radii (centred-window target rates with a shrinkage prior),
at multiple quantisation scales (a division ladder pushed through target encoding), and
through classic exact-match smoothing keys — all **cross-fitted** so that no row ever
sees its own label.

### Feature Families at a Glance

| Family | What it captures | Leakage-safe mechanism |
|--------|------------------|------------------------|
| Digit features | Generator rounding / synthetic fingerprints | Deterministic transform |
| Generator-lift features | pool frequency / original frequency | Never touches `y` |
| Smooth keys + quantisation ladder | Multi-resolution value identity | Inner-fold cross-fitted TE |
| Neighbourhood windows | Local target rate around each income/commute value | Inner-fold cross-fitted, prior-shrunk |
| Triple Target Encoding | `auto` / 10 / 100 smoothing per categorical | sklearn `TargetEncoder(cv=...)` |
| Original-data target means | Real-world priors per category | Fit on original dataset only |

### Key Design Choices
- **10-fold CV with the original 10K rows concatenated per fold** — the real-data rows act as a clean prior anchor.
- **Histogram + prefix-sum window scan** — O(n) centred-window target rates instead of a slow sort-merge.
- **Additive prior of 10 pseudo-counts** toward the fold-train mean keeps sparse windows stable.

# 1. Problem Statement

<div style="background:#f0fdfa; padding:20px; border-radius:10px; border-left:4px solid #10b981;">

<h3 style="margin-top:0;">Objective</h3>

Predict whether a consumer will buy an electric vehicle, based on demographics, income, commuting behaviour, charging infrastructure and attitude features.

</div>

---

## Business Context

EV adoption is the cornerstone of the transport-energy transition. For OEMs, utilities and policymakers,
correctly identifying likely EV buyers means:

- Targeting **incentives and subsidy campaigns** at the right audience
- Planning **charging infrastructure** where prospective owners actually live and commute
- Understanding the **range-anxiety barrier** and how it interacts with income
- Tuning **environmental messaging** to the segments that respond to it

---

## Evaluation Metric

<div style="background:#ecfdf5; padding:20px; border-radius:10px; border-left:4px solid #10b981;">

<h4 style="margin-top:0;">ROC-AUC (Receiver Operating Characteristic – Area Under Curve)</h4>

ROC-AUC measures the model's ability to rank a random buyer above a random non-buyer, across all classification thresholds.

</div>

### Interpretation

| AUC Score | Interpretation |
|-----------|---------------|
| **0.50** | Random guessing |
| **0.70 – 0.80** | Acceptable model |
| **0.80 – 0.90** | Strong predictive performance |
| **> 0.90** | Excellent discrimination |

On this competition the leaderboard lives above **0.94**, so the marginal features — and their
leak-free construction — are what separate submissions.

## Machine Learning Pipeline

<div style="background: #f8fafc; padding: 30px; border-radius: 12px; text-align: center; font-family: monospace; font-size: 14px;">

```
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│  DATA LOADING   │ ──▶│       EDA       │ ──▶│    CLEANING     │ ──▶│    FEATURES     │
│  Train/Test/Orig│    │  Distributions  │    │  Target 0/1 map │    │ Digit/Lift/Keys │
└─────────────────┘    └─────────────────┘    └─────────────────┘    └─────────────────┘
                                                                              │
                                                                              ▼
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│   SUBMISSION    │ ◀──│   GPU XGBOOST   │ ◀──│   TRIPLE-TE     │ ◀──│    WINDOWS      │
│  CSV + OOF      │    │ 10-Fold + ES    │    │ auto/10/100     │    │ Cross-fitted    │
└─────────────────┘    └─────────────────┘    └─────────────────┘    └─────────────────┘
```

</div>

<div class="alert alert-info" style="border-radius: 8px; padding: 15px; margin-top: 20px; background: #ecfeff; border: 1px solid #22d3ee;">
    <b style="color: #155e75;">Leakage discipline:</b> <span style="color: #164e63;">Every target-aware encoding (windows, group windows, ladder TE, triple TE) is cross-fitted on an inner StratifiedKFold of the fold-training rows only. Validation and test rows receive full fold-training statistics — never their own label.</span>
</div>

## 2. Dataset Description and Loading

### Data Sources
- **Training Set**: 668,665 synthetic consumer rows with the `Will_Buy_EV` label
- **Test Set**: 286,571 unlabeled rows to predict
- **Original Dataset**: 10,000 real-world rows (EV Adoption and Range Anxiety dataset) — concatenated into every fold's training data and used for target-free lift features

### Features Overview
| Category | Features | Description |
|----------|----------|-------------|
| **Demographics** | Age, Gender, Annual_Income_USD | Customer profile |
| **Mobility** | Daily_Commute_km, Number_of_Cars_Owned | Travel behaviour |
| **Infrastructure** | Charging_Stations_Near_Home, Charging_Stations_Near_Work, Home_Charging_Possible | Charging access |
| **Attitudes** | Environmental_Concern_Level, Range_Anxiety_Level, Subsidy_Available | Psychological / economic drivers |
| **Vehicle** | City_Type, Current_Car_Type | Current car ownership context |
| **Target** | Will_Buy_EV | Intends to buy an EV (Yes/No) |

In [ ]:
# Import Libraries
import os
import gc
import time
import random
import warnings
import numpy as np
import pandas as pd
from sklearn import __version__ as sklearn_version
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score
import xgboost as xgb

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
# Configuration
class CFG:
    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e9/train.csv"
    TEST_PATH  = "/kaggle/input/competitions/playground-series-s6e9/test.csv"
    ORIG_PATH  = "/kaggle/input/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety/EV_Adoption_and_Range_Anxiety_Dataset.csv"

    TARGET = 'Will_Buy_EV'

    N_FOLDS = 10
    RANDOM_SEED = 42

    NUM_ROUND = 8000
    ES_ROUNDS = 400

    LIFT_CATS = ['Environmental_Concern_Level', 'Range_Anxiety_Level', 'Subsidy_Available',
                 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible']

# Neighbourhood encoding-block constants (module level, used inside the fold loop)
WIN_INC  = [2, 5, 10, 25, 50, 200]
WIN_KM   = [1, 3, 10]
WIN_GRP  = [5, 25]
WIN_PRIOR = 10.0
TE_SMOOTHS = [('auto', 'auto'), (10.0, '10'), (100.0, '100')]

# Seed everything
def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)

seed_everything(CFG.RANDOM_SEED)

# XGBoost Parameters
XGB_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'booster': 'gbtree',
    'tree_method': 'hist',
    'device': 'cuda',
    'seed': CFG.RANDOM_SEED,
    'nthread': -1,
    'verbosity': 0,
    'learning_rate': 0.01,
    'max_depth': 3,
    'max_leaves': 8,
    'grow_policy': 'lossguide',
    'min_child_weight': 4.532387806880492,
    'subsample': 0.7400402414525654,
    'colsample_bytree': 0.85,
    'alpha': 0.7523885021652775,
    'reg_lambda': 0.6189949691705282,
    'gamma': 1.0,
    'max_bin': 1024,
}

# Metric
def auc_score(y_true, y_probs):
    """ROC AUC for binary classification."""
    return roc_auc_score(y_true, y_probs)

print("Configuration set!")
print(f"  N_FOLDS: {CFG.N_FOLDS}")
print(f"  NUM_ROUND / ES_ROUNDS: {CFG.NUM_ROUND} / {CFG.ES_ROUNDS}")
print(f"  device: {XGB_PARAMS['device']}")

In [ ]:
# Load Data
print("Loading datasets...")
train = pd.read_csv(CFG.TRAIN_PATH)
test  = pd.read_csv(CFG.TEST_PATH)
orig  = pd.read_csv(CFG.ORIG_PATH)

# Encode target variable
target2idx = {'No': 0, 'Yes': 1}
if not pd.api.types.is_numeric_dtype(train[CFG.TARGET]):
    train[CFG.TARGET] = (train[CFG.TARGET].astype(str).str.strip()
                         .str.title().map(target2idx))
if not pd.api.types.is_numeric_dtype(orig[CFG.TARGET]):
    orig[CFG.TARGET] = (orig[CFG.TARGET].astype(str).str.strip()
                        .str.title().map(target2idx))

# Store IDs for later
train_id = train['id'].copy()
test_id  = test['id'].copy()
y_orig   = orig[CFG.TARGET].copy().reset_index(drop=True)

print(f"Train Shape : {train.shape}")
print(f"Test Shape  : {test.shape}")
print(f"Orig Shape  : {orig.shape}")

print(f"\nTarget mapping: {target2idx}")
print("\nClass Distribution (train):")
class_counts = train[CFG.TARGET].value_counts().sort_index()
for cls, count in class_counts.items():
    print(f"  Class {cls}: {count:,} ({100*count/len(train):.1f}%)")
print(f"Pos rate (train): {train[CFG.TARGET].mean():.4f}")
print(f"Pos rate (orig):  {orig[CFG.TARGET].mean():.4f}")

## 3. Exploratory Data Analysis (EDA)

Let's explore the dataset to understand the distributions, relationships, and the
generator artifacts we will later encode as features.

In [ ]:
# Dataset Overview
print("="*60)
print("TRAINING DATA OVERVIEW")
print("="*60)
display(train.head())

print("\n" + "="*60)
print("DATA TYPES")
print("="*60)
print(train.dtypes.to_frame('dtype'))

In [ ]:
# Missing Values Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_missing = train.isnull().sum()
train_missing = train_missing[train_missing > 0]
if len(train_missing) > 0:
    axes[0].barh(train_missing.index, train_missing.values, color='coral')
    axes[0].set_xlabel('Missing Count')
    axes[0].set_title('Missing Values in Training Set')
else:
    axes[0].text(0.5, 0.5, 'No Missing Values!', ha='center', va='center', fontsize=14)
    axes[0].set_title('Missing Values in Training Set')

num_stats_cols = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
                  'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
                  'Environmental_Concern_Level']
stats_df = train[num_stats_cols].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]
axes[1].axis('off')
table = axes[1].table(cellText=stats_df.round(2).values,
                      rowLabels=stats_df.index,
                      colLabels=stats_df.columns,
                      cellLoc='center',
                      loc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)
axes[1].set_title('Numerical Features Statistics', fontsize=12, pad=20)

plt.tight_layout()
plt.show()

In [ ]:
# Target Distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

train_target = train[CFG.TARGET].value_counts().sort_index()
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(train_target.values, labels=['No', 'Yes'], autopct='%1.1f%%',
            colors=colors, explode=[0, 0.05], startangle=90)
axes[0].set_title('Training Set - Target Distribution')

orig_target = orig[CFG.TARGET].value_counts().sort_index()
axes[1].pie(orig_target.values, labels=['No', 'Yes'], autopct='%1.1f%%',
            colors=colors, explode=[0, 0.05], startangle=90)
axes[1].set_title('Original Dataset - Target Distribution')

x = np.arange(2)
width = 0.35
axes[2].bar(x - width/2, [train_target[0], orig_target[0]], width, label='No', color='#2ecc71')
axes[2].bar(x + width/2, [train_target[1], orig_target[1]], width, label='Yes', color='#e74c3c')
axes[2].set_xticks(x)
axes[2].set_xticklabels(['Training', 'Original'])
axes[2].set_ylabel('Count')
axes[2].set_title('Target Distribution Comparison')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"EV-buy rate - Train: {train[CFG.TARGET].mean()*100:.2f}%")
print(f"EV-buy rate - Orig:  {orig[CFG.TARGET].mean()*100:.2f}%")

In [ ]:
# Numerical Features Distribution by Target
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, col in enumerate(['Annual_Income_USD', 'Daily_Commute_km', 'Age']):
    for val, label, color in [(0, 'No', '#2ecc71'), (1, 'Yes', '#e74c3c')]:
        data = train[train[CFG.TARGET] == val][col]
        axes[idx].hist(data, bins=60, alpha=0.6, label=label, color=color, density=True)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Density')
    axes[idx].set_title(f'{col} Distribution by Target')
    axes[idx].legend()

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("  - Annual_Income_USD: buyers skew heavily toward the high-income tail;")
print("    the raw dollar axis is the single most information-dense column.")
print("  - Daily_Commute_km: longer commutes correlate with lower purchase intent")
print("    (range-anxiety pressure).")
print("  - Age: mild effect, mostly mediated through income and car ownership.")

In [ ]:
# Categorical Features Analysis
CAT_DISPLAY = ['Environmental_Concern_Level', 'Range_Anxiety_Level', 'Subsidy_Available',
               'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for idx, col in enumerate(CAT_DISPLAY):
    rate = train.groupby(col)[CFG.TARGET].mean().sort_values(ascending=False)
    cmap = plt.cm.RdYlGn_r
    bars = axes[idx].bar(range(len(rate)), rate.values,
                         color=cmap(np.linspace(0.1, 0.9, len(rate))))
    axes[idx].set_xticks(range(len(rate)))
    axes[idx].set_xticklabels(rate.index, rotation=45, ha='right', fontsize=8)
    axes[idx].set_ylabel('EV-Buy Rate')
    axes[idx].set_title(f'{col}', fontsize=11)
    axes[idx].axhline(y=train[CFG.TARGET].mean(), color='black', linestyle='--', alpha=0.5)
    for bar, val in zip(bars, rate.values):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                       f'{val:.2f}', ha='center', va='bottom', fontsize=7)

axes[7].axis('off')
plt.suptitle('EV-Buy Rate by Categorical Features', fontsize=15, y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
# The Continuous Axes: Income x Commute Structure
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sample = train.sample(min(50000, len(train)), random_state=CFG.RANDOM_SEED)
for val, label, color in [(0, 'No', '#2ecc71'), (1, 'Yes', '#e74c3c')]:
    s = sample[sample[CFG.TARGET] == val]
    axes[0].scatter(s['Daily_Commute_km'], s['Annual_Income_USD'],
                    s=2, alpha=0.25, label=label, color=color)
axes[0].set_yscale('log')
axes[0].set_xlabel('Daily_Commute_km')
axes[0].set_ylabel('Annual_Income_USD (log)')
axes[0].set_title('Income x Commute (50K sample, log income)')
axes[0].legend()

# Income histogram with generator artifacts highlighted
axes[1].hist(train['Annual_Income_USD'], bins=300, color='#0ea5e9', alpha=0.8)
axes[1].axvline(x=30000, color='purple', linestyle='--', linewidth=2, label='30K spike')
axes[1].axvspan(38000, 42000, color='red', alpha=0.2, label='dead zone (38K-42K)')
axes[1].axvline(x=170537, color='orange', linestyle='--', linewidth=2, label='millionaire cliff')
axes[1].set_xlabel('Annual_Income_USD')
axes[1].set_ylabel('Count')
axes[1].set_title('Income Distribution - CTGAN Generator Artifacts')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nCTGAN-generated data leaves fingerprints: a value spike at 30,000,")
print("a near-empty dead zone around 38,000-42,000, and a hard cliff at 170,537.")
print("Later we encode these exact bounds as flags and lift features.")

In [ ]:
# Correlation Matrix for Numerical Features
num_display = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
               'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
               'Environmental_Concern_Level', CFG.TARGET]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

corr_matrix = train[num_display].corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', center=0, ax=axes[0],
            fmt='.2f', square=True)
axes[0].set_title('Numerical Features Correlation')

pivot = train.pivot_table(values=CFG.TARGET, index='Range_Anxiety_Level',
                          columns='Environmental_Concern_Level', aggfunc='mean')
sns.heatmap(pivot, annot=True, cmap='RdYlGn_r', center=train[CFG.TARGET].mean(),
            ax=axes[1], fmt='.3f')
axes[1].set_title('EV-Buy Rate: Range Anxiety x Environmental Concern')

plt.tight_layout()
plt.show()

print("\nNote: the Range-Anxiety x Environmental-Concern cell grid is strongly")
print("non-additive - the interaction justifies bigram and cross features.")

### Key Insights from EDA

<div class="alert alert-success" style="border-radius: 8px; padding: 20px; background: #ecfdf5; border: 1px solid #10b981;">

<h4 style="margin-top:0; color:#047857;">Key Findings</h4>

<ul>

<li><strong>Generator fingerprints everywhere:</strong> as CTGAN output, the training data contains a value spike at $30,000 income, a sparse dead zone around $38,000-$42,000, and a hard millionaire cliff at $170,537 — all exploitable as explicit flags.</li>

<li><strong>Income is the dominant axis:</strong> EV-buy rate rises steeply with income, with an apparent decision bound near $41,667 — this motivates multi-scale encodings of the raw dollar value.</li>

<li><strong>Commute distance cuts the other way:</strong> longer daily commutes associate with lower purchase intent, modulated by charging-station density near work.</li>

<li><strong>Attitudes interact:</strong> Environmental Concern Level and Range Anxiety Level are individually informative but their 5x3 grid is strongly non-additive — low anxiety + high concern is the classic "EV recipe".</li>

<li><strong>Class balance is moderate:</strong> the positive rate sits near one third, so ROC-AUC on well-calibrated probabilities is the right target; no resampling tricks needed.</li>

<li><strong>The original 10K real rows differ subtly from the synthetic pool</strong> in category frequencies — the ratio between pool frequency and original frequency (generator "lift") is itself a signal.</li>

</ul>

</div>

## 4. Feature Engineering

### Feature Engineering Pipeline

Every transform below is applied identically to train, test and the original dataset.
Target-aware encodings are deferred to the fold loop so they can be cross-fitted.

| Step | Feature Family | Description |
|------|----------------|-------------|
| 1 | Digit Features | Digits of each numeric after x1e4 int64 rescale — synthetic-data fingerprints |
| 2 | Engineered Interactions | ECL x Subsidy, ECL x Range-Anxiety, Income x Subsidy, logs, hard-edge flags |
| 3 | Artifact & Structural Flags | 30K spike, dead zone, millionaire cliff, buy bound |
| 4 | Smooth Keys + Quantisation Ladder | Exact/100/1000 floor keys plus `//10, //50, //500, //5000, //5, //50` string keys |
| 5 | Generator-Lift Features | pool freq / orig freq per category & key, plus novelty flag — never touches `y` |
| 6 | Original Target Means | Per-value target means from the real 10K rows mapped onto train/test |
| 7 | Numeric-as-String + Frequency Encoding | Micro-values get their own TE category; pool frequencies |
| 8 | Targeted Bigrams | ECL_bin x RangeAnxiety, ECL_bin x Subsidy |
| 9 | Selective Groupby Deviations | z-deviation of numerics inside attitude/income groups |
| 10 | Redundancy Prune | Drop constants and perfectly-correlated columns |
| 11 | Window Encodings (in fold loop) | Cross-fitted centred-window target rates for income, commute, group-restricted |
| 12 | Triple Target Encoding (in fold loop) | Same categorical set at `auto` / 10 / 100 smoothing |

In [ ]:
# Feature-engineering helpers (1/4): digits, engineered, artifacts, structure, keys
def add_digit_features(df, num_cols):
    """Digit features from a single int64 rescale."""
    df = df.copy()

    for c in num_cols:
        scaled = np.rint(df[c].fillna(0).astype('float64') * 1e4).astype('int64')
        for k in range(-4, 4):
            df[f"{c}_digit{k}"] = ((scaled // np.int64(10 ** (k + 4))) % 10).astype('int8')

    return df

def add_engineered_features(df):
    """EDA-validated engineered features: 4 interactions, 3 logs, 4 hard-edge flags."""
    df = df.copy()

    for col in ['Annual_Income_USD', 'Daily_Commute_km', 'Environmental_Concern_Level']:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    ra_map = {'Low': 0, 'Medium': 1, 'High': 2}
    df['_RA_code'] = df['Range_Anxiety_Level'].map(ra_map).fillna(0).astype('int8')

    df['_Subsidy_bin'] = (df['Subsidy_Available'] == 'Yes').astype('int8')

    df['_ECL_x_Subsidy'] = (
        df['Environmental_Concern_Level'] * df['_Subsidy_bin']
    ).astype('float32')
    df['_ECL_x_RangeAnxiety'] = (
        df['Environmental_Concern_Level'] * (3 - df['_RA_code'])
    ).astype('float32')
    df['_Income_x_Subsidy'] = (
        df['Annual_Income_USD'] * df['_Subsidy_bin']
    ).astype('float32')
    df['_Charging_Total'] = (
        df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work']
    ).astype('float32')

    df['_log_Income'] = np.log1p(df['Annual_Income_USD'].clip(lower=0)).astype('float32')
    df['_log_Commute'] = np.log1p(df['Daily_Commute_km'].clip(lower=0)).astype('float32')
    df['_log_Charging_Total'] = np.log1p(df['_Charging_Total'].clip(lower=0)).astype('float32')

    df['_high_income'] = (df['Annual_Income_USD'] > 170537).astype('int8')
    df['_range_anxiety_high'] = (df['Range_Anxiety_Level'] == 'High').astype('int8')
    df['_ecl_max'] = (df['Environmental_Concern_Level'] == 5).astype('int8')
    df['_ev_recipe'] = (
        (df['Environmental_Concern_Level'] == 5) &
        (df['Range_Anxiety_Level'] == 'Low')
    ).astype('int8')

    df = df.drop(columns=['_RA_code', '_Subsidy_bin'])

    return df

def add_synthetic_artifact_flags(df):
    """CTGAN anomaly flags for the generated distribution."""
    df = df.copy()
    df['is_30k_spike'] = (df['Annual_Income_USD'] == 30000.0).astype('int8')
    df['is_millionaire_cliff'] = (df['Annual_Income_USD'] >= 170537.0).astype('int8')
    df['is_dead_zone'] = ((df['Annual_Income_USD'] >= 38000.0) &
                          (df['Annual_Income_USD'] <= 42000.0)).astype('int8')
    df['is_env_hater'] = (df['Environmental_Concern_Level'] == 1).astype('int8')
    return df

def add_structural_flags(df):
    """Exact structural bounds decoded from the generator recipe (buy bound, dead zone)."""
    df = df.copy()
    df['_below_buy_bound'] = (df['Annual_Income_USD'] < 41667).astype('int8')
    df['_dead_zone_exact'] = ((df['Annual_Income_USD'] >= 31004) &
                              (df['Annual_Income_USD'] <= 41970)).astype('int8')
    return df

def add_smooth_keys(df):
    """Multi-scale 'Smooth Keys' (binned numerics as strings)."""
    df = df.copy()
    df['income_exact_int'] = np.floor(df['Annual_Income_USD']).astype(str)
    df['income100_floor']  = np.floor(df['Annual_Income_USD'] / 100.0).astype(str)
    df['income1000_floor'] = np.floor(df['Annual_Income_USD'] / 1000.0).astype(str)
    df['commute_integer']  = np.floor(df['Daily_Commute_km']).astype(str)
    return df

In [ ]:
# Feature-engineering helpers (2/4): lift features and original-dataset priors
def add_lift_features(train_df, test_df, orig_df):
    """Generator-lift features, lift = freq_pool / freq_orig; never touches y."""
    print("   Adding generator-lift features (pool freq / orig freq, target-free)...")
    new_cols = []

    def _lift(keys_tr, keys_te, keys_og, col_name):
        pool = pd.concat([keys_tr, keys_te], ignore_index=True)
        comp_freq = pool.value_counts(normalize=True)
        orig_freq = keys_og.value_counts(normalize=True)
        for df_, keys in ((train_df, keys_tr), (test_df, keys_te), (orig_df, keys_og)):
            raw = keys.map(comp_freq).fillna(0.0) / keys.map(orig_freq)
            df_[col_name] = raw.fillna(0.0).astype('float32')
        new_cols.append(col_name)

    _lift(train_df['income_exact_int'], test_df['income_exact_int'],
          orig_df['income_exact_int'], 'lift_inc_exact')
    _lift(train_df['income100_floor'], test_df['income100_floor'],
          orig_df['income100_floor'], 'lift_income100')
    _lift(train_df['commute_integer'], test_df['commute_integer'],
          orig_df['commute_integer'], 'lift_commute')
    for c in CFG.LIFT_CATS:
        _lift(train_df[c].astype(str), test_df[c].astype(str),
              orig_df[c].astype(str), f'lift_{c}')

    for df_ in (train_df, test_df, orig_df):
        df_['_tri_key'] = (df_['Environmental_Concern_Level'].fillna(3).astype(int).astype(str)
                           + '_' + df_['Subsidy_Available'].astype(str)
                           + '_' + df_['Range_Anxiety_Level'].astype(str))
    _lift(train_df['_tri_key'], test_df['_tri_key'], orig_df['_tri_key'], 'lift_trigram')

    orig_inc_set = set(orig_df['income_exact_int'].unique())
    train_df['novel_inc'] = (~train_df['income_exact_int'].isin(orig_inc_set)).astype('int8')
    test_df['novel_inc']  = (~test_df['income_exact_int'].isin(orig_inc_set)).astype('int8')
    orig_df['novel_inc']  = np.int8(0)
    new_cols.append('novel_inc')

    for df_ in (train_df, test_df, orig_df):
        df_.drop(columns=['_tri_key'], errors='ignore', inplace=True)

    print(f"      Added {len(new_cols)} lift/novelty columns: {new_cols}")
    return train_df, test_df, orig_df, new_cols

def add_original_target_means(train_df, test_df, orig_df, cat_cols, num_cols, target):
    """Map the original dataset's per-value target means onto train and test."""
    orig_global_mean = orig_df[target].mean()
    orig_stats = {}
    for col in cat_cols + num_cols:
        if col in orig_df.columns:
            orig_stats[col] = orig_df.groupby(col, observed=False)[target].mean()

    for col, stats in orig_stats.items():
        map_name = f"{col}_org_mean"
        train_df[map_name] = train_df[col].map(stats).fillna(orig_global_mean).astype('float32')
        test_df[map_name]  = test_df[col].map(stats).fillna(orig_global_mean).astype('float32')

    return train_df, test_df

In [ ]:
# Feature-engineering helpers (3/4): string keys, frequency encoding, bigrams, groupby
def add_numeric_as_string(df, num_cols):
    """Numeric-as-string columns so micro-values get their own TE category."""
    df = df.copy()
    num_to_cat_cols = []
    for col in num_cols:
        if col not in df.columns:
            continue
        cat_name = f"{col}_cat"
        df[cat_name] = df[col].fillna('NaN').astype(str)
        num_to_cat_cols.append(cat_name)
    return df, num_to_cat_cols

def add_frequency_encoding(train_df, test_df, cols_to_encode):
    """Frequency encoding computed on the train+test pool (no target used)."""
    combined = pd.concat([train_df[cols_to_encode], test_df[cols_to_encode]], axis=0)
    for col in cols_to_encode:
        freq_mapping = combined[col].value_counts(normalize=True).to_dict()
        train_df[f"{col}_fe"] = train_df[col].map(freq_mapping).astype('float32').fillna(0.0)
        test_df[f"{col}_fe"]  = test_df[col].map(freq_mapping).astype('float32').fillna(0.0)
    return train_df, test_df

def add_targeted_bigrams(train_df, test_df):
    """Targeted bigram interactions: ECL_bin x RA, ECL_bin x Subsidy."""
    print("   Adding TARGETED bigram interactions (ECL_bin x RA, ECL_bin x Subsidy)...")

    for df in [train_df, test_df]:
        df['ECL_bin'] = df['Environmental_Concern_Level'].fillna(3).astype(int).astype(str)

    bigram_name_1 = 'bigram_ECL_bin_x_RangeAnxiety'
    train_df[bigram_name_1] = train_df['ECL_bin'] + '_' + train_df['Range_Anxiety_Level'].astype(str)
    test_df[bigram_name_1]  = test_df['ECL_bin'] + '_' + test_df['Range_Anxiety_Level'].astype(str)

    bigram_name_2 = 'bigram_ECL_bin_x_Subsidy'
    train_df[bigram_name_2] = train_df['ECL_bin'] + '_' + train_df['Subsidy_Available'].astype(str)
    test_df[bigram_name_2]  = test_df['ECL_bin'] + '_' + test_df['Subsidy_Available'].astype(str)

    bigram_cols = [bigram_name_1, bigram_name_2]
    return train_df, test_df, bigram_cols

def add_selective_groupby(train_df, test_df):
    """Selective groupby deviations; pool stats, never the target."""
    print("   Adding SELECTIVE groupby deviation features (ECL_bin, income_bin groups)...")

    if 'ECL_bin' not in train_df.columns:
        for df in [train_df, test_df]:
            df['ECL_bin'] = df['Environmental_Concern_Level'].fillna(3).astype(int).astype(str)

    for df in [train_df, test_df]:
        df['income_bin'] = pd.cut(df['Annual_Income_USD'].fillna(df['Annual_Income_USD'].median()),
                                  bins=5, labels=['1', '2', '3', '4', '5']).astype(str)

    combined = pd.concat([train_df, test_df], axis=0)
    group_cols = ['ECL_bin', 'income_bin', 'Subsidy_Available', 'Range_Anxiety_Level']
    agg_num_cols = ['Annual_Income_USD', 'Environmental_Concern_Level', 'Daily_Commute_km']

    new_features = []
    for group_col in group_cols:
        for num_col in agg_num_cols:
            if group_col == num_col or group_col not in combined.columns:
                continue
            grp_mean = combined.groupby(group_col, observed=False)[num_col].mean()
            grp_std  = combined.groupby(group_col, observed=False)[num_col].std().fillna(0)
            dev_name = f'grp_{group_col}_{num_col}_dev'
            train_df[dev_name] = ((train_df[num_col] - train_df[group_col].map(grp_mean)) /
                                  (train_df[group_col].map(grp_std) + 1e-6)).astype('float32')
            test_df[dev_name]  = ((test_df[num_col] - test_df[group_col].map(grp_mean)) /
                                  (test_df[group_col].map(grp_std) + 1e-6)).astype('float32')
            new_features.append(dev_name)

    print(f"      Added {len(new_features)} SELECTIVE groupby deviation features")
    return train_df, test_df

def drop_redundant_features(train_df, test_df, target):
    """Drop constant columns and columns perfectly correlated (|r| = 1.0)."""
    eval_cols = [c for c in train_df.columns
                 if c not in ['id', target] and pd.api.types.is_numeric_dtype(train_df[c])]
    to_drop_corr = []
    if len(eval_cols) > 1:
        corr_matrix = train_df[eval_cols].corr().abs()
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop_corr = [column for column in upper_tri.columns if any(upper_tri[column] == 1.0)]
        del corr_matrix, upper_tri
        gc.collect()

    to_drop_const = [c for c in train_df.columns if train_df[c].nunique() == 1] + \
                    [c for c in test_df.columns if test_df[c].nunique() == 1]

    DROP = set(to_drop_corr).union(set(to_drop_const))
    DROP = [c for c in DROP if c not in ['id', target]]

    if len(DROP) > 0:
        print(f"   Dropping {len(DROP)} redundant/constant features")
        train_df = train_df.drop(columns=DROP, errors='ignore')
        test_df  = test_df.drop(columns=DROP, errors='ignore')
    else:
        print("   No redundant features found.")

    return train_df, test_df, DROP

def align_original_schema(orig_df):
    """Align original dataset schema with competition train."""
    keep_cols = [
        'Age', 'Annual_Income_USD', 'Daily_Commute_km',
        'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
        'Charging_Stations_Near_Work', 'Environmental_Concern_Level',
        'Gender', 'City_Type', 'Current_Car_Type',
        'Home_Charging_Possible', 'Subsidy_Available',
        'Range_Anxiety_Level',
    ]
    return orig_df[keep_cols].copy()

In [ ]:
# Neighbourhood encoding block: quantisation ladder + cross-fitted window rates
LADDER_COLS = ['ladder_inc10', 'ladder_inc50', 'ladder_inc500', 'ladder_inc5000',
               'ladder_km5', 'ladder_km50']

def add_quantisation_ladder(df):
    """Quantisation keys as strings, pushed through the target encoder at all three smoothings."""
    inc = np.rint(df['Annual_Income_USD'].astype('float64')).astype('int64')
    km = np.rint(df['Daily_Commute_km'].astype('float64')).astype('int64')
    for name, v, d in [('inc10', inc, 10), ('inc50', inc, 50), ('inc500', inc, 500),
                       ('inc5000', inc, 5000), ('km5', km, 5), ('km50', km, 50)]:
        df[f'ladder_{name}'] = (v // d).astype(str)
    return df

def window_rates(v_fit, y_fit, v_query, widths, prior, gm):
    """Centred-window smoothed target rate: histogram + prefix-sum range scan."""
    lo, hi = int(min(v_fit.min(), v_query.min())), int(max(v_fit.max(), v_query.max()))
    n = hi - lo + 1
    cnt = np.bincount(v_fit - lo, minlength=n).astype(np.float64)
    s   = np.bincount(v_fit - lo, weights=y_fit, minlength=n)
    ccnt = np.concatenate([[0.0], np.cumsum(cnt)]); cs = np.concatenate([[0.0], np.cumsum(s)])
    q = v_query - lo; out = []
    for w in widths:
        a = np.clip(q - w, 0, n); b = np.clip(q + w + 1, 0, n)
        out.append((((cs[b]-cs[a]) + prior*gm) / ((ccnt[b]-ccnt[a]) + prior)).astype("float32"))
    return np.stack(out, 1)

def group_window_scan(g_a, inc_a, y_a, g_q, inc_q, gm):
    """Income windows restricted to the City_Type x Current_Car_Type group."""
    out = np.full((len(inc_q), len(WIN_GRP)), gm, np.float32)
    for g in np.unique(g_q):
        ma, mq = g_a == g, g_q == g
        if ma.sum(): out[mq] = window_rates(inc_a[ma], y_a[ma], inc_q[mq], WIN_GRP, WIN_PRIOR, gm)
    return out

def crossfit_windows(key_fit, y_fit, key_val, key_test, widths, prior, gm, prefix):
    """One window family: inner StratifiedKFold(5) on fit rows, full stats for val/test."""
    fit_cols = np.zeros((len(key_fit), len(widths)), np.float32)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG.RANDOM_SEED)
    for tr_i, q_i in skf.split(key_fit, y_fit):
        fit_cols[q_i] = window_rates(key_fit[tr_i], y_fit[tr_i], key_fit[q_i], widths, prior, gm)
    val_cols = window_rates(key_fit, y_fit, key_val, widths, prior, gm)
    test_cols = window_rates(key_fit, y_fit, key_test, widths, prior, gm)
    names = [f'win_{prefix}_{w}' for w in widths]
    return {nm: (fit_cols[:, j], val_cols[:, j], test_cols[:, j]) for j, nm in enumerate(names)}

def crossfit_group_windows(g_fit, inc_fit, y_fit, g_val, inc_val, g_test, inc_test, gm):
    """Group-restricted income windows under the same cross-fitting discipline."""
    fit_cols = np.zeros((len(inc_fit), len(WIN_GRP)), np.float32)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG.RANDOM_SEED)
    for tr_i, q_i in skf.split(g_fit, y_fit):
        fit_cols[q_i] = group_window_scan(g_fit[tr_i], inc_fit[tr_i], y_fit[tr_i], g_fit[q_i], inc_fit[q_i], gm)
    val_cols = group_window_scan(g_fit, inc_fit, y_fit, g_val, inc_val, gm)
    test_cols = group_window_scan(g_fit, inc_fit, y_fit, g_test, inc_test, gm)
    names = [f'win_grp_inc_{w}' for w in WIN_GRP]
    return {nm: (fit_cols[:, j], val_cols[:, j], test_cols[:, j]) for j, nm in enumerate(names)}

In [ ]:
# 0. Define Feature Groups and Scan Keys
med_inc, med_km = train['Annual_Income_USD'].median(), train['Daily_Commute_km'].median()
def int_key(df, col, scale, med):
    """Integer scan key: raw dollars (scale 1) or tenths of a km (scale 10)."""
    return np.rint(df[col].fillna(med).astype('float64') * scale).astype('int64').values
def group_key(df):
    return df['City_Type'].astype(str) + '_' + df['Current_Car_Type'].astype(str)

inc_tr, inc_te, inc_or = (int_key(d, 'Annual_Income_USD', 1.0, med_inc) for d in (train, test, orig))
km_tr, km_te, km_or = (int_key(d, 'Daily_Commute_km', 10.0, med_km) for d in (train, test, orig))
g_codes = pd.factorize(pd.concat([group_key(train), group_key(test), group_key(orig)],
                                 ignore_index=True))[0].astype('int32')
grp_tr, grp_te, grp_or = g_codes[:len(train)], g_codes[len(train):len(train)+len(test)], g_codes[len(train)+len(test):]

train = train.drop(columns=['id'])
test  = test.drop(columns=['id'])
orig_aligned = align_original_schema(orig).reset_index(drop=True)

print(f"   Train shape: {train.shape}")
print(f"   Test shape:  {test.shape}")
print(f"   Orig shape:  {orig_aligned.shape}")

CATS = [c for c in test.columns if train[c].dtype == object or train[c].dtype.name == 'str']
NUMS = [c for c in test.columns if c not in CATS]

print(f"   Categorical columns ({len(CATS)}): {CATS}")
print(f"   Numerical columns ({len(NUMS)}): {NUMS}")

print(f"\n   Target mapping: {target2idx}")
print(f"   Pos rate (train): {train[CFG.TARGET].mean():.4f}")
print(f"   Pos rate (orig):  {y_orig.mean():.4f}")
print(f"   Orig share of the training matrix: {len(orig_aligned)/(len(train)+len(orig_aligned)):.4%}")
print(f"   Rows held out per fold: {len(train)//CFG.N_FOLDS:,}")
print(f"   Missing in orig: {int(orig_aligned.isna().sum().sum())} (XGBoost handles NaN natively)")

In [ ]:
# 1. Digit Features
print("[2/5] Feature Engineering (full pipeline + neighbourhood encoding block)...")

print("   Adding digit features...")
train = add_digit_features(train, NUMS)
test  = add_digit_features(test,  NUMS)
orig_aligned = add_digit_features(orig_aligned, NUMS)

In [ ]:
# 2. Engineered Interactions + Artifact & Structural Flags
print("   Adding engineered interactions + flags...")
train = add_engineered_features(train)
test  = add_engineered_features(test)
orig_aligned = add_engineered_features(orig_aligned)

print("   Adding synthetic-artifact flags...")
train = add_synthetic_artifact_flags(train)
test  = add_synthetic_artifact_flags(test)
orig_aligned = add_synthetic_artifact_flags(orig_aligned)

print("   Adding STRUCTURAL flags (buy bound 41,667 + exact dead zone)...")
train = add_structural_flags(train)
test  = add_structural_flags(test)
orig_aligned = add_structural_flags(orig_aligned)

In [ ]:
# 3. Smooth Keys + Quantisation Ladder
print("   Adding multi-scale smooth keys + quantisation ladder...")
train = add_smooth_keys(add_quantisation_ladder(train))
test  = add_smooth_keys(add_quantisation_ladder(test))
orig_aligned = add_smooth_keys(add_quantisation_ladder(orig_aligned))

In [ ]:
# 4. Original Target Means
print("   Adding original dataset target means...")
train, test = add_original_target_means(train, test, orig, CATS, NUMS, CFG.TARGET)
orig_global_mean = orig[CFG.TARGET].mean()
for col in CATS + NUMS:
    org_mean_name = f"{col}_org_mean"
    if org_mean_name in train.columns and col in orig_aligned.columns:
        orig_stats = orig.groupby(col, observed=False)[CFG.TARGET].mean()
        orig_aligned[org_mean_name] = orig_aligned[col].map(orig_stats).fillna(orig_global_mean).astype('float32')
    elif org_mean_name in train.columns:
        orig_aligned[org_mean_name] = orig_global_mean
        orig_aligned[org_mean_name] = orig_aligned[org_mean_name].astype('float32')

In [ ]:
# 5. Generator-Lift Features
train, test, orig_aligned, lift_cols = add_lift_features(train, test, orig_aligned)

In [ ]:
# 6. Numeric-as-String + Frequency Encoding
print("   Converting numerics to string categories...")
all_numeric_cols = [c for c in train.columns
                    if c != CFG.TARGET and pd.api.types.is_numeric_dtype(train[c])
                    and not c.startswith('_') and not c.startswith('is_')]
engineered_numeric = ['_ECL_x_Subsidy', '_ECL_x_RangeAnxiety', '_Income_x_Subsidy',
                      '_Charging_Total', '_log_Income', '_log_Commute', '_log_Charging_Total']
engineered_numeric = [c for c in engineered_numeric if c in train.columns]
all_numeric_to_str = all_numeric_cols + engineered_numeric

train, train_num_cat_cols = add_numeric_as_string(train, all_numeric_to_str)
test, _ = add_numeric_as_string(test, all_numeric_to_str)
orig_aligned, _ = add_numeric_as_string(orig_aligned, all_numeric_to_str)
print(f"   Created {len(train_num_cat_cols)} numeric-as-string columns")

print("   Adding frequency encoding on all cat + num-as-string cols...")
freq_target_cols = CATS + train_num_cat_cols + [
    'income_exact_int', 'income100_floor', 'income1000_floor', 'commute_integer'
]
freq_target_cols = [c for c in freq_target_cols if c in train.columns and c in test.columns]
train, test = add_frequency_encoding(train, test, freq_target_cols)
for col in freq_target_cols:
    if f"{col}_fe" in train.columns:
        orig_aligned[f"{col}_fe"] = 0.0

In [ ]:
# 7. Targeted Bigrams
train, test, bigram_cols = add_targeted_bigrams(train, test)
orig_aligned['ECL_bin'] = orig_aligned['Environmental_Concern_Level'].fillna(3).astype(int).astype(str)
for bigram_name in bigram_cols:
    if 'ECL_bin_x_RangeAnxiety' in bigram_name:
        orig_aligned[bigram_name] = orig_aligned['ECL_bin'] + '_' + orig_aligned['Range_Anxiety_Level'].astype(str)
    elif 'ECL_bin_x_Subsidy' in bigram_name:
        orig_aligned[bigram_name] = orig_aligned['ECL_bin'] + '_' + orig_aligned['Subsidy_Available'].astype(str)

In [ ]:
# 8. Selective Groupby Deviations
train, test = add_selective_groupby(train, test)
orig_aligned['income_bin'] = pd.cut(
    orig_aligned['Annual_Income_USD'].fillna(orig_aligned['Annual_Income_USD'].median()),
    bins=5, labels=['1', '2', '3', '4', '5']).astype(str)
combined_for_grp = pd.concat([
    train[['ECL_bin', 'income_bin', 'Subsidy_Available', 'Range_Anxiety_Level',
           'Annual_Income_USD', 'Environmental_Concern_Level', 'Daily_Commute_km']],
    test[['ECL_bin', 'income_bin', 'Subsidy_Available', 'Range_Anxiety_Level',
          'Annual_Income_USD', 'Environmental_Concern_Level', 'Daily_Commute_km']]],
    axis=0)
group_cols = ['ECL_bin', 'income_bin', 'Subsidy_Available', 'Range_Anxiety_Level']
agg_num_cols = ['Annual_Income_USD', 'Environmental_Concern_Level', 'Daily_Commute_km']
for group_col in group_cols:
    for num_col in agg_num_cols:
        if group_col == num_col or group_col not in combined_for_grp.columns:
            continue
        grp_mean = combined_for_grp.groupby(group_col, observed=False)[num_col].mean()
        grp_std  = combined_for_grp.groupby(group_col, observed=False)[num_col].std().fillna(0)
        dev_name = f'grp_{group_col}_{num_col}_dev'
        orig_aligned[dev_name] = ((orig_aligned[num_col] - orig_aligned[group_col].map(grp_mean)) /
                                  (orig_aligned[group_col].map(grp_std) + 1e-6)).astype('float32')

In [ ]:
# 9. Redundancy Prune + Encoding Column Sets
for df in [train, test, orig_aligned]:
    df.drop(columns=['ECL_bin', 'income_bin'], errors='ignore', inplace=True)

print("   Feature selection (drop constants + perfectly-correlated)...")
train, test, dropped = drop_redundant_features(train, test, CFG.TARGET)
orig_aligned = orig_aligned.drop(columns=[c for c in dropped if c in orig_aligned.columns], errors='ignore')

TARGET_ENCODE_COLS = [c for c in (CATS + train_num_cat_cols + [
    'income_exact_int', 'income100_floor', 'income1000_floor', 'commute_integer'
] + bigram_cols + LADDER_COLS) if c in train.columns]

FEATURES = [c for c in test.columns if c != 'id']

print(f"\n   Total features: {len(FEATURES)}")
print(f"   Columns to Triple-TE: {len(TARGET_ENCODE_COLS)}")
print(f"     - Generator-lift / novelty cols: {len([c for c in lift_cols if c in FEATURES])}")
print(f"     - Targeted bigram cols: {len([c for c in bigram_cols if c in TARGET_ENCODE_COLS])}")
print(f"   Window encodings per fold: {len(WIN_INC)} income-window + {len(WIN_KM)} commute-window + "
      f"{len(WIN_GRP)} group-window cols, {len(LADDER_COLS) * len(TE_SMOOTHS)} ladder TE cols = "
      f"{len(WIN_INC) + len(WIN_KM) + len(WIN_GRP) + len(LADDER_COLS) * len(TE_SMOOTHS)} new cols")

## 5. Model Training

### Per-Fold Anatomy

<div style="background: #fef3c7; padding: 20px; border-radius: 10px; border-left: 4px solid #f59e0b; margin: 15px 0;">

<h4 style="margin-top: 0; color: #92400e;">Inside every fold:</h4>
<ul style="color: #78350f;">
<li><b>Concatenate the original fold-training slice</b> of the real 10K rows (each fold sees its own 9K of them under an independent KFold split).</li>
<li><b>Cross-fit the neighbourhood windows</b>: income in raw dollars at radii 2/5/10/25/50/200, commute in tenths of a km at 1/3/10, and the same income windows restricted to City_Type x Current_Car_Type groups at radii 5/25 — all computed by histogram + prefix-sum range scan with an additive prior of 10 pseudo-counts toward the fold-train mean, fitted through an inner StratifiedKFold(5) of the fold-training rows.</li>
<li><b>Triple Target Encoding</b>: the same categorical + ladder column set encoded at smoothings <code>auto</code>, <code>10</code> and <code>100</code>, so the trees can pick the resolution they trust.</li>
<li><b>Train shallow GPU XGBoost</b> with early stopping and predict validation + test.</li>
</ul>

</div>

### XGBoost Configuration

| Parameter | Value | Parameter | Value |
|-----------|-------|-----------|-------|
| learning_rate | 0.01 | max_depth / max_leaves | 3 / 8 (`lossguide`) |
| subsample | 0.7400 | colsample_bytree | 0.85 |
| min_child_weight | 4.5324 | reg_alpha / reg_lambda | 0.7524 / 0.6190 |
| gamma | 1.0 | max_bin | 1024 |
| device | cuda | rounds / early stopping | 8000 / 400 |
| CV | KFold(10, shuffle, seed=42) | metric | ROC-AUC |

In [ ]:
# [3/5] Train XGBoost (10-Fold CV, GPU, orig concat + Triple TE + windows)
t0_all = time.time()

print(f"\n[3/5] Training XGBoost ({CFG.N_FOLDS}-Fold CV, GPU, orig concat + Triple TE + windows)...")

X      = train.drop([CFG.TARGET], axis=1)
y      = train[CFG.TARGET]
test_X = test.copy()
y_np   = y.values

oof_probs  = np.zeros(len(y))
test_probs = np.zeros(len(test_X))
best_iters = []
fold_scores = []
n_te = 0
cols_used = 0

kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.RANDOM_SEED)
kf_orig = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.RANDOM_SEED)
orig_splits = list(kf_orig.split(orig_aligned, y_orig))

for fold, ((train_idx, val_idx), (or_train_idx, or_val_idx)) in enumerate(
        zip(kf.split(X), orig_splits)):

    fold_start = time.time()
    print(f"\n   Fold {fold+1}/{CFG.N_FOLDS}:")

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    orig_tr = orig_aligned.iloc[or_train_idx].copy()
    y_orig_tr = y_orig.iloc[or_train_idx].copy()
    X_train = pd.concat([X_train, orig_tr], axis=0).reset_index(drop=True)
    y_train = pd.concat([y_train, y_orig_tr], axis=0).reset_index(drop=True)
    X_test_fold = test_X.copy()

    inc_a = np.concatenate([inc_tr[train_idx], inc_or[or_train_idx]])
    km_a  = np.concatenate([km_tr[train_idx], km_or[or_train_idx]])
    grp_a = np.concatenate([grp_tr[train_idx], grp_or[or_train_idx]])
    y_fit = y_train.to_numpy(dtype=np.float64); gm = float(y_fit.mean())
    window_blocks = {}
    window_blocks.update(crossfit_windows(inc_a, y_fit, inc_tr[val_idx], inc_te, WIN_INC, WIN_PRIOR, gm, 'inc'))
    window_blocks.update(crossfit_windows(km_a, y_fit, km_tr[val_idx], km_te, WIN_KM, WIN_PRIOR, gm, 'km'))
    window_blocks.update(crossfit_group_windows(grp_a, inc_a, y_fit, grp_tr[val_idx], inc_tr[val_idx], grp_te, inc_te, gm))
    for nm, (c_fit, c_val, c_test) in window_blocks.items():
        X_train[nm], X_val[nm], X_test_fold[nm] = c_fit, c_val, c_test

    te_feature_names = []
    for smooth_val, smooth_name in TE_SMOOTHS:
        te = TargetEncoder(target_type='binary', smooth=smooth_val,
                           cv=CFG.N_FOLDS, shuffle=True, random_state=CFG.RANDOM_SEED)
        X_train_enc = te.fit_transform(X_train[TARGET_ENCODE_COLS], y_train).astype('float32')
        X_val_enc   = te.transform(X_val[TARGET_ENCODE_COLS]).astype('float32')
        X_test_enc  = te.transform(X_test_fold[TARGET_ENCODE_COLS]).astype('float32')

        for i, col in enumerate(TARGET_ENCODE_COLS):
            te_name = f"TE_{col}_{smooth_name}"
            X_train[te_name] = X_train_enc[:, i]
            X_val[te_name]   = X_val_enc[:, i]
            X_test_fold[te_name] = X_test_enc[:, i]
            if te_name not in te_feature_names:
                te_feature_names.append(te_name)

    n_te = len(te_feature_names)

    cols_to_drop_after_te = [c for c in TARGET_ENCODE_COLS if c in X_train.columns]
    X_train = X_train.drop(columns=cols_to_drop_after_te, errors='ignore')
    X_val   = X_val.drop(columns=cols_to_drop_after_te, errors='ignore')
    X_test_fold = X_test_fold.drop(columns=cols_to_drop_after_te, errors='ignore')

    string_cols = [c for c in X_train.columns if not pd.api.types.is_numeric_dtype(X_train[c])]
    if string_cols:
        X_train = X_train.drop(columns=string_cols)
        X_val   = X_val.drop(columns=string_cols)
        X_test_fold = X_test_fold.drop(columns=string_cols)

    cols_fold = list(X_train.columns)
    cols_used = len(cols_fold)
    if fold == 0:
        print(f"      Feature count: {len(cols_fold)} "
              f"({n_te} Triple TE of {len(TARGET_ENCODE_COLS)} cols, "
              f"{len(window_blocks)} window cols) | "
              f"train rows {len(train_idx):,} comp + {len(or_train_idx):,} orig")

    dtrain = xgb.DMatrix(X_train[cols_fold], label=y_train, feature_names=cols_fold)
    dval   = xgb.DMatrix(X_val[cols_fold], label=y_val, feature_names=cols_fold)
    dtest  = xgb.DMatrix(X_test_fold[cols_fold], feature_names=cols_fold)
    booster = xgb.train(XGB_PARAMS, dtrain, num_boost_round=CFG.NUM_ROUND,
                        evals=[(dval, 'valid')], verbose_eval=False,
                        callbacks=[xgb.callback.EarlyStopping(rounds=CFG.ES_ROUNDS,
                                                              save_best=True)])
    best_iter = booster.num_boosted_rounds()

    val_probs = booster.predict(dval)
    oof_probs[val_idx] = val_probs
    test_probs += booster.predict(dtest) / CFG.N_FOLDS

    best_iters.append(best_iter)
    fold_auc = auc_score(y_val.values, val_probs)
    fold_scores.append(fold_auc)
    print(f"      AUC: {fold_auc:.5f} | BestIter {best_iter} | "
          f"Time {(time.time()-fold_start)/60:.1f} min | Total {(time.time()-t0_all)/60:.1f} min")

    if fold == 0:
        try:
            imp = booster.get_score(importance_type='gain')
            top15 = sorted(imp.items(), key=lambda kv: kv[1], reverse=True)[:15]
            print("      Fold-1 top-15 gain importances:")
            for _name, _score in top15:
                print(f"        {_name:<46s} {_score:.1f}")
        except Exception as _e:
            print(f"      (importance diagnostics skipped: {_e})")

    del dtrain, dval, dtest, booster, X_train, X_val, X_test_fold
    gc.collect()

cv_score = auc_score(y_np, oof_probs)
print(f"\n   {'='*70}")
print(f"   {CFG.N_FOLDS}-Fold OOF CV (AUC): {cv_score:.6f}")
print(f"   Fold mean {np.mean(fold_scores):.6f} +/- {np.std(fold_scores):.6f}")
print(f"   Best iterations: {best_iters}")

## 6. Evaluation & Diagnostics

Overall CV metrics, per-fold behaviour, the OOF ROC curve and the test-prediction distribution.

In [ ]:
# Final results summary
print(f"\n{'='*80}")
print(f"[5/5] RESULTS - Single Model, {CFG.N_FOLDS}-Fold XGBoost (GPU, window encodings + Triple TE)")
print(f"{'='*80}")
print(f"OOF CV (AUC): {cv_score:.6f}")
print(f"Fold mean {np.mean(fold_scores):.6f} +/- {np.std(fold_scores):.6f}")
print(f"Best iterations: {best_iters}")
print(f"Columns used: {cols_used} ({len(FEATURES)} base + {n_te} Triple TE + {len(window_blocks)} window cols)")
print(f"Test predictions: min {test_probs.min():.6f} | max {test_probs.max():.6f}")
print(f"\nTotal time: {(time.time()-t0_all)/60:.1f} min")
print(f"{'='*80}")

In [ ]:
# Visualization: Fold Performance and OOF ROC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
x_pos = np.arange(1, CFG.N_FOLDS + 1)
bars = ax1.bar(x_pos, fold_scores, color='#10b981', alpha=0.85, edgecolor='#065f46')
ax1.axhline(y=np.mean(fold_scores), color='#e74c3c', linestyle='--', linewidth=1.5,
            label=f'Mean {np.mean(fold_scores):.5f}')
ax1.set_xticks(x_pos)
ax1.set_xlabel('Fold')
ax1.set_ylabel('AUC Score')
ax1.set_title('Validation AUC per Fold', fontsize=12, fontweight='bold')
ax1.set_ylim([min(fold_scores) - 0.002, max(fold_scores) + 0.002])
for bar, s in zip(bars, fold_scores):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
             f'{s:.4f}', ha='center', va='bottom', fontsize=8)
ax1.legend()

ax2 = axes[1]
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_np, oof_probs)
ax2.plot(fpr, tpr, color='#0e7490', linewidth=2, label=f'OOF (AUC = {cv_score:.5f})')
ax2.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
ax2.fill_between(fpr, tpr, alpha=0.1, color='#0e7490')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve (Out-of-Fold)', fontsize=12, fontweight='bold')
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# OOF and Test Prediction Distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(oof_probs[y_np == 0], bins=100, alpha=0.6, label='Negative',
             color='#2ecc71', density=True)
axes[0].hist(oof_probs[y_np == 1], bins=100, alpha=0.6, label='Positive',
             color='#e74c3c', density=True)
axes[0].set_xlabel('OOF Predicted Probability')
axes[0].set_ylabel('Density')
axes[0].set_title('OOF Score Separation by Class', fontsize=12, fontweight='bold')
axes[0].legend()

axes[1].hist(test_probs, bins=100, color='#0ea5e9', edgecolor='black', alpha=0.7)
axes[1].axvline(x=np.mean(test_probs), color='red', linestyle='--', linewidth=2,
                label=f'Mean = {np.mean(test_probs):.4f}')
axes[1].axvline(x=np.mean(oof_probs), color='green', linestyle='-', linewidth=2,
                label=f'OOF mean = {np.mean(oof_probs):.4f}')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Count')
axes[1].set_title('Test Set Prediction Distribution', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nTest Set Statistics:")
print(f"  Mean prediction: {np.mean(test_probs):.4f}")
print(f"  Std prediction:  {np.std(test_probs):.4f}")
print(f"  Min prediction:  {np.min(test_probs):.4f}")
print(f"  Max prediction:  {np.max(test_probs):.4f}")
print(f"  OOF mean:        {np.mean(oof_probs):.4f}  (train pos rate: {y_np.mean():.4f})")

## 7. Saving Predictions

Save the Out-of-Fold predictions and the test submission (probabilities, as required by ROC-AUC scoring).

In [ ]:
# Save OOF and submission files
out_dir = "/kaggle/working"
os.makedirs(out_dir, exist_ok=True)

oof_df = pd.DataFrame({'id': train_id, 'pred': oof_probs})
oof_df.to_csv(f"{out_dir}/oof.csv", index=False)
print(f"   [SAVED] {out_dir}/oof.csv (id, pred)  - probabilities")

sub_df = pd.DataFrame({'id': test_id, CFG.TARGET: test_probs})
sub_df.to_csv(f"{out_dir}/submission.csv", index=False)
print(f"   [SAVED] {out_dir}/submission.csv  - probabilities, "
      f"range {test_probs.min():.4f} .. {test_probs.max():.4f}")

print("\nSubmission preview:")
display(sub_df.head())
print("OOF preview:")
display(oof_df.head())

### Key Success Factors

<div class="alert alert-success" style="border-radius: 8px; padding: 20px; margin: 15px 0; background: #ecfdf5; border: 1px solid #10b981;">
    <h4 style="margin-top: 0; color: #047857;">What Made This Solution Effective</h4>
    <ul style="color: #065f46; margin-bottom: 0;">
        <li><strong>Neighbourhood Window Encodings:</strong> centred-window target rates at 9 income radii, 3 commute radii and 2 group-restricted scales — turning two continuous axes into a multi-resolution local prior</li>
        <li><strong>Triple Target Encoding:</strong> the same columns at auto/10/100 smoothing so the trees choose their own bias-variance tradeoff</li>
        <li><strong>Generator-Aware Features:</strong> digit decomposition, lift ratios, structural flags and novelty indicators exploit the CTGAN fingerprints directly</li>
        <li><strong>Strict Cross-Fitting:</strong> inner StratifiedKFold(5) on fold-training rows only; validation and test get full fold-training statistics</li>
        <li><strong>Shallow, Heavily-Regularized GPU XGBoost:</strong> depth 3 / 8 leaves with 8000 rounds and patience-400 early stopping</li>
    </ul>
</div>

### Future Improvements an Estimate(Not Confirmed)

<div class="alert alert-warning" style="border-radius: 8px; padding: 20px; margin: 15px 0; background: #fffbeb; border: 1px solid #f59e0b;">
    <h4 style="margin-top: 0; color: #92400e;">Potential Enhancements</h4>
    <ul style="color: #78350f; margin-bottom: 0;">
        <li>Add LightGBM / CatBoost members and blend with rank-averaging</li>
        <li>Learn the optimal window radii instead of fixing a geometric ladder</li>
        <li>Joint 2-D windows over (income, commute) as a single key</li>
        <li>Optuna re-tuning of the depth/leaves/regularization block</li>
    </ul>
</div>

<div style="background: linear-gradient(135deg, #0f172a 0%, #134e4a 100%); padding: 30px; border-radius: 12px; margin-top: 20px;">

<h3 style="color: #ffffff; margin-top: 0;">Solution Summary</h3>

<p style="color: #94a3b8; font-size: 14px;">
<strong style="color: #34d399;">Model:</strong> Single GPU XGBoost, 10-fold CV, original-data rows concatenated per fold
</p>

<p style="color: #94a3b8; font-size: 14px;">
<strong style="color: #34d399;">Key Innovation:</strong> Cross-fitted neighbourhood window encodings on the two continuous axes — histogram + prefix-sum range scans with an additive prior — plus a quantisation ladder pushed through triple target encoding
</p>

<p style="color: #94a3b8; font-size: 14px;">
<strong style="color: #34d399;">Feature Engineering:</strong> digit features, generator-lift ratios, structural artifact flags, smooth keys, bigrams, groupby deviations, frequency encoding and original-data target means
</p>

<p style="color: #94a3b8; font-size: 14px;">
<strong style="color: #34d399;">Validation:</strong> 10-fold CV with inner 5-fold cross-fitting for every target-aware transform — the OOF score is honest
</p>

<hr style="border: none; height: 1px; background: #374151; margin: 20px 0;">

<p style="color: #e2e8f0; font-size: 14px;">
On synthetic playground data the marginal signal hides in <em>how a value is generated</em>, not just in the value itself.
By giving the model the same axis viewed at many neighbourhood radii and quantisation scales — each one computed leak-free —
a single shallow model can compete with far more complicated stacks.
</p>

<p style="color: #64748b; font-size: 13px; text-align: right; margin-bottom: 0;">If this notebook helped you, an upvote is appreciated. Happy modeling!</p>

</div>